# Лекція 5. Локальний деплоймент і оптимізація моделей

Демо до лекції. Усі моделі в цьому ноутбуці запускаються **локально** через [Ollama](https://ollama.com), їх використання відбувається через HTTP-запити.

## 0. Налаштування середовища

### 0.0. Імпортуємо бібліотеки

In [ ]:
import json
import subprocess
import sys
import time
import urllib.request

OLLAMA_URL = "http://localhost:11434"

### 0.1. Встановлюємо й запускаємо Ollama

Локально цю клітинку можна просто виконати: вона нічого не зробить.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !sudo apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

### 0.2. Перевіряємо, що Ollama працює

Ollama — це HTTP-сервер на порту 11434. Функція `ollama()` надсилає йому запит і повертає відповідь як словник Python; якщо сервер працює, побачимо його версію.

In [ ]:
def ollama(path, body=None, timeout=900):
    """POST-запит до Ollama (або GET, якщо body=None); повертає розібраний JSON."""
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(OLLAMA_URL + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.load(resp)


print("Ollama version:", ollama("/api/version")["version"])

## 1. Base, Instruct і Thinking

Порівнюємо три поведінки однієї й тієї самої моделі Qwen3 1.7B:
- **Base-модель** пройшла лише попереднє тренування: вона вміє передбачати наступний токен в інтернет-текстах, але не вміє відповідати на питання як асистент.
- **Instruct-модель** додатково донавчили на прикладах «інструкція → відповідь» у форматі діалогу.
- **Thinking** — режим, у якому Instruct-модель перед відповіддю генерує ланцюжок міркувань.

### 1.0. Завантажуємо моделі

Беремо одну й ту саму модель Qwen3 1.7B у двох версіях:
- **Base** — [Qwen3-1.7B-Base](https://huggingface.co/Qwen/Qwen3-1.7B-Base) у форматі GGUF (Q4_K_M) з Hugging Face;
- **Instruct** — офіційна [qwen3:1.7b](https://ollama.com/library/qwen3) з каталогу Ollama; вона ж уміє працювати в режимі міркування.

Перше завантаження займає кілька хвилин.

In [ ]:
BASE_MODEL = "hf.co/mradermacher/Qwen3-1.7B-Base-GGUF:Q4_K_M"
INSTRUCT_MODEL = "qwen3:1.7b"

for m in (BASE_MODEL, INSTRUCT_MODEL):
    print("pull", m, "→", ollama("/api/pull", {"model": m, "stream": False})["status"])

### 1.1. Допоміжні функції

Нам вистачить двох ендпойнтів Ollama:
- `/api/generate` з `"raw": true` — передає моделі текст як є, **без chat template**;
- `/api/chat` — приймає список повідомлень і сам загортає їх у chat template моделі.

У Qwen3 режим міркування вмикається параметром `think`. З `think=True` модель спершу генерує ланцюжок міркувань (Ollama повертає його окремим полем `message.thinking`), а вже потім — відповідь. Функція `compare_thinking()` запускає запит в обох режимах і показує відповідь, кількість згенерованих токенів і час.

In [ ]:
def generate_raw(model, prompt, max_tokens=120):
    """Продовження тексту без chat template — так «бачить» запит Base-модель."""
    r = ollama("/api/generate", {
        "model": model, "prompt": prompt, "raw": True, "stream": False,
        "options": {"temperature": 0, "seed": 1, "num_predict": max_tokens},
    })
    return r["response"]


def chat(model, prompt, think=False, max_tokens=3000):
    """Один запит у форматі діалогу; think=True вмикає режим міркування."""
    return ollama("/api/chat", {
        "model": model, "messages": [{"role": "user", "content": prompt}],
        "think": think, "stream": False,
        "options": {"temperature": 0, "seed": 1, "num_predict": max_tokens},
    })


def compare_thinking(prompt):
    for think in (False, True):
        r = chat(INSTRUCT_MODEL, prompt, think=think)
        msg = r["message"]
        print(f"=== think={think} | токенів: {r['eval_count']} | генерація: {r['eval_duration'] / 1e9:.1f} с")
        if msg.get("thinking"):
            print(f"--- міркування ({len(msg['thinking'])} символів), початок:")
            print(msg["thinking"][:600], "…")
        print("--- відповідь:")
        print(msg["content"])
        print()

### 1.2. Base: модель просто продовжує текст

Поставимо Base- та Instruct-моделі однакове питання.

In [ ]:
QUESTION = "Яка столиця Франції?"

In [ ]:
print(generate_raw(BASE_MODEL, QUESTION))

Для Base-моделі «Яка столиця Франції?» схоже на початок списку питань вікторини — тож вона продовжує цей список, а не відповідає.

### 1.3. Instruct: модель відповідає на питання

In [ ]:
print(chat(INSTRUCT_MODEL, QUESTION)["message"]["content"])

### 1.4. Instruct проти Thinking: багатокрокова задача

In [ ]:
compare_thinking(
    "У кошику 3 яблука. Марія поклала туди ще 2 пакети по 4 яблука, "
    "а потім зʼїла 2 яблука. Скільки яблук у кошику?"
)

Обидва режими відповідають правильно, але з міркуванням модель генерує в кілька разів більше токенів — а користувач бачить відповідь лише після них. Зверніть увагу: міркує модель англійською, хоча питання українською.

### 1.5. Міркування не гарантують правильності

In [ ]:
compare_thinking("Скільки разів літера «р» трапляється в слові «трансформер»?")

Правильна відповідь — 3 (т**р**ансфо**р**ме**р**), але мала модель помиляється в обох режимах. Чому так — подумайте самі (це питання до лабораторної роботи).

Спробуйте власні запити: де режим міркування справді допомагає, а де лише витрачає токени?

## 2. Квантизація з Quanto

Завантажимо ту саму модель у чотирьох варіантах — від оригінальних 16 біт до 2 біт — і подивимося, що квантизація змінює в памʼяті, швидкості та якості. Якість перевіримо трьома способами: перплексією, перекладом з автоматичними метриками й очима — на вільній генерації тексту.

**Quanto.** [Optimum Quanto](https://github.com/huggingface/optimum-quanto) — бібліотека квантизації від Hugging Face, один із бекендів [Optimum](https://huggingface.co/docs/optimum). У Transformers вона підключається через [`QuantoConfig`](https://huggingface.co/docs/transformers/quantization/quanto) і квантизує модель прямо під час завантаження, тому окремий файл з квантизованими вагами не потрібен. Квантизуються ваги лінійних шарів (`nn.Linear`) — у них більшість параметрів моделі, а ембединги, нормалізації та вихідний шар лишаються в 16 бітах.

| Варіант | Біт на вагу | Що це |
|---|---|---|
| fp16 | 16 | Оригінальні ваги, точка відліку |
| int8 | 8 | 8-бітні цілі числа з масштабом |
| int4 | 4 | 4-бітні цілі числа, ваги діляться на блоки з власними масштабами |
| int2 | 2 | 2-бітні цілі числа — лише 4 можливі значення на вагу; тут модель має зламатися |

**Обчислення.** Множити числа в цих форматах напряму T4 не вміє, тому перед множенням ваги розпаковуються в 16 біт. Тому памʼяті потрібно менше, а от генерація часто **не прискорюється**, а то й сповільнюється.

**Де запускати.** Секцію запускаємо **в Google Colab на GPU T4** (Runtime → Change runtime type → T4 GPU). Беремо формат **fp16**, а не bf16: T4 не підтримує bf16 апаратно, а для запуску готової моделі різниця між ними практично непомітна.

### 2.0. Встановлюємо бібліотеки й перевіряємо GPU

In [ ]:
%pip uninstall -y -q diffusers  # optimum-quanto імпортує diffusers, якщо він є; передвстановлена в Colab версія несумісна з його torchao
%pip install -q "transformers==5.20.1" "accelerate==1.15.0" "optimum-quanto==0.2.7" "evaluate==0.4.6" "sacrebleu==2.6.0"

In [ ]:
import gc
import time
from types import SimpleNamespace

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from optimum.quanto.library.extensions import get_extension
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig

assert torch.cuda.is_available(), "Увімкніть GPU: Runtime → Change runtime type → T4 GPU"
DEVICE = "cuda"
DTYPE = torch.float16
print("GPU:", torch.cuda.get_device_name())


def unpack_weights(packed, bits):
    """Розпаковує 4- чи 2-бітні значення, по кілька упакованих в одному байті."""
    values_per_byte = 8 // bits
    mask = 2**bits - 1
    return torch.cat([(packed >> (bits * i)) & mask for i in range(values_per_byte)]).to(torch.uint8)


# Quanto на GPU компілює власні CUDA-ядра, розраховані на A100 і новіші; на T4 ця компіляція падає.
# На T4 з усіх цих ядер потрібна лише розпаковка int4/int2-ваг, тож підставляємо її версію на чистому PyTorch.
if torch.cuda.get_device_capability()[0] < 8:
    get_extension("quanto_cuda")._lib = SimpleNamespace(unpack=unpack_weights)

### 2.1. Модель і варіанти квантизації

Беремо [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) — сильнішу модель того ж сімейства, що й у секції 1: 4B параметрів, лише Instruct-режим без міркувань. У fp16 вона займає ~8 GB і вміщається в T4 з його 16 GB.

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

VARIANTS = {
    "fp16": None,
    "int8": QuantoConfig(weights="int8"),
    "int4": QuantoConfig(weights="int4"),
    "int2": QuantoConfig(weights="int2"),
}

### 2.2. Що міряємо

**Перплексія** — стандартна автоматична метрика якості квантизації (нею міряє, зокрема, llama.cpp). Модель читає фіксований текст, і для кожного токена ми дивимося, яку ймовірність вона йому давала. Перплексія — це, грубо кажучи, з якої кількості варіантів модель у середньому «вагалася»: чим вона менша, тим краще модель знає мову й тему. Еталонних відповідей тут не потрібно, а число змінюється плавно, тому видно навіть невеликі втрати. Порівнювати перплексію можна лише в межах однієї мови: англійська й українська токенізуються по-різному.

**Переклад EN → UK** — реалістичне завдання з генерацією тексту, для якого є еталонні переклади. Порівнюємо переклад моделі з еталоном двома стандартними метриками з бібліотеки [Evaluate](https://huggingface.co/docs/evaluate):
- [**chrF**](https://github.com/huggingface/evaluate/blob/main/metrics/chrf/README.md) — наскільки збігаються послідовності символів (0–100);
- [**BLEU**](https://github.com/huggingface/evaluate/tree/main/metrics/sacrebleu) — наскільки збігаються послідовності слів (0–100). Сувора метрика: синонім чи інший порядок слів уже знижують бал.

**Вільна генерація** — лист, пояснення, план, код. Тут еталону немає, тож оцінюємо очима в 2.5.

Крім якості, для кожного варіанта записуємо **памʼять моделі** на GPU, **пік VRAM** і **швидкість генерації** (tokens/s).

In [ ]:
PPL_TEXT = {
    "English": (
        "Photosynthesis is the process by which green plants, algae and some bacteria turn light energy into "
        "chemical energy. It takes place mainly in the leaves, inside small structures called chloroplasts, which "
        "contain the green pigment chlorophyll. Chlorophyll absorbs sunlight, and the plant uses this energy to "
        "combine carbon dioxide from the air with water from the soil. The result is glucose, a sugar that the "
        "plant uses for growth and as a store of energy, and oxygen, which is released into the atmosphere. "
        "Almost all life on Earth depends on photosynthesis: it produces the oxygen we breathe and forms the "
        "basis of most food chains."
    ),
    "Українська": (
        "Фотосинтез — це процес, під час якого зелені рослини, водорості та деякі бактерії перетворюють енергію "
        "світла на хімічну енергію. Він відбувається переважно в листках, усередині маленьких структур, які "
        "називають хлоропластами і які містять зелений пігмент хлорофіл. Хлорофіл поглинає сонячне світло, і "
        "рослина використовує цю енергію, щоб поєднати вуглекислий газ із повітря з водою із ґрунту. У результаті "
        "утворюються глюкоза — цукор, який рослина використовує для росту і як запас енергії, — та кисень, що "
        "виділяється в атмосферу. Майже все життя на Землі залежить від фотосинтезу: він дає кисень, яким ми "
        "дихаємо, і лежить в основі більшості харчових ланцюгів."
    ),
}

TRANSLATION = [
    # (речення англійською, еталонний переклад)
    ("The meeting has been moved to Thursday at three o'clock.", "Зустріч перенесли на четвер на третю годину."),
    ("Please send me the report by the end of the day.", "Будь ласка, надішліть мені звіт до кінця дня."),
    ("The library is closed on Sundays and public holidays.", "Бібліотека не працює в неділю та у святкові дні."),
    ("Our team fixed the bug, but the tests are still failing.", "Наша команда виправила помилку, але тести все ще не проходять."),
    ("It is going to rain tomorrow, so take an umbrella.", "Завтра буде дощ, тож візьміть парасольку."),
    ("The new model is smaller, but it answers almost as well as the old one.", "Нова модель менша, але відповідає майже так само добре, як стара."),
    ("I would like to book a table for two for seven o'clock tonight.", "Я хотів би забронювати столик на двох на сьому вечора."),
    ("The price includes breakfast and free Wi-Fi.", "У вартість входять сніданок і безкоштовний Wi-Fi."),
    ("Students must submit their lab work before the deadline.", "Студенти мають здати лабораторну роботу до кінцевого терміну."),
    ("If the server does not respond, restart it and check the logs.", "Якщо сервер не відповідає, перезапустіть його й перевірте логи."),
]
TRANSLATE_PROMPT = "Translate the following sentence into Ukrainian. Output only the translation.\n\n{}"

GEN_PROMPTS = [
    "Напиши короткий лист викладачеві з проханням перенести здачу лабораторної роботи на тиждень через хворобу.",
    "Поясни студентові першого курсу в 3–4 реченнях, що таке квантизація нейромережі.",
    "Explain in three sentences why the sky is blue.",
    "Склади план підготовки до іспиту з програмування на 5 днів, по одному пункту на день.",
    "Write a Python function that checks whether a string is a palindrome.",
]

### 2.3. Проганяємо всі варіанти

Дві допоміжні функції — перплексія тексту й генерація відповіді — і цикл по варіантах: завантажуємо модель, міряємо все потрібне, звільняємо памʼять. Займає кілька хвилин на варіант.

In [ ]:
def perplexity(model, text):
    """exp(середня похибка передбачення наступного токена) на фіксованому тексті."""
    ids = tokenizer(text, return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        loss = model(ids, labels=ids).loss
    return float(torch.exp(loss))


def generate(model, prompt, max_new_tokens):
    """Відповідь без випадковості; повертає текст, кількість нових токенів і час генерації."""
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.cuda.synchronize()
    new_tokens = output[0, inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), len(new_tokens), time.perf_counter() - start


records = {}
for name, config in VARIANTS.items():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map=DEVICE, quantization_config=config)

    r = {"памʼять моделі, GB": torch.cuda.memory_allocated() / 1e9}
    for lang, text in PPL_TEXT.items():
        r[f"перплексія, {lang}"] = perplexity(model, text)
    r["переклади"] = [generate(model, TRANSLATE_PROMPT.format(src), 80)[0] for src, _ in TRANSLATION]
    generations = [generate(model, prompt, 200) for prompt in GEN_PROMPTS]
    r["генерації"] = [text for text, _, _ in generations]
    r["tokens/s"] = sum(n for _, n, _ in generations) / sum(s for _, _, s in generations)
    r["пік VRAM, GB"] = torch.cuda.max_memory_allocated() / 1e9
    records[name] = r
    print(f"{name}: готово")

    del model

### 2.4. Памʼять, швидкість і автоматичні метрики

In [ ]:
chrf = evaluate.load("chrf")
bleu = evaluate.load("sacrebleu")
references = [[ref] for _, ref in TRANSLATION]

summary = pd.DataFrame({
    name: {
        "памʼять моделі, GB": r["памʼять моделі, GB"],
        "пік VRAM, GB": r["пік VRAM, GB"],
        "tokens/s": r["tokens/s"],
        **{f"перплексія, {lang}": r[f"перплексія, {lang}"] for lang in PPL_TEXT},
        "chrF перекладу": chrf.compute(predictions=r["переклади"], references=references)["score"],
        "BLEU перекладу": bleu.compute(predictions=r["переклади"], references=references)["score"],
    }
    for name, r in records.items()
}).T.round(2)
summary

Перплексія для кожної мови окремо. Вісь — логарифмічна: інакше на одному графіку не помістилися б і кілька одиниць, і сотні.

In [ ]:
LANG_COLORS = {"English": "#2a78d6", "Українська": "#eb6834"}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

variants = list(records)
x = np.arange(len(variants))
width = 0.38

fig, ax = plt.subplots(figsize=(9, 4.2))
for i, lang in enumerate(PPL_TEXT):
    values = summary[f"перплексія, {lang}"].values
    bars = ax.bar(x + (i - 0.5) * width, values, width - 0.04, color=LANG_COLORS[lang], label=lang)
    ax.bar_label(bars, labels=[f"{v:.1f}" for v in values], fontsize=8, color=MUTED, padding=2)
ax.set_yscale("log")
ax.set_xticks(x, variants)
ax.set_ylabel("перплексія (менше — краще)", color=INK)
ax.set_title("Перплексія на фіксованому тексті", loc="left", color=INK)
ax.legend(frameon=False, ncols=2, loc="upper left")
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.show()

Переклади всіх варіантів поруч з еталоном:

In [ ]:
translations = pd.DataFrame({"еталон": [ref for _, ref in TRANSLATION]}, index=[src for src, _ in TRANSLATION])
for name, r in records.items():
    translations[name] = r["переклади"]
translations

### 2.5. Сліпе порівняння: fp16 проти int4

Автоматичні метрики показали, наскільки int4 відрізняється від fp16 у числах. Тепер перевірмо очима — так, як це роблять оцінювачі-люди: для кожного запиту показуємо дві відповіді, fp16 та int4, **у випадковому порядку й без підписів**. Ваше завдання — обрати кращу.

In [ ]:
rng = np.random.default_rng(0)
BLIND_ORDER = [rng.permutation(["fp16", "int4"]) for _ in GEN_PROMPTS]

for i, prompt in enumerate(GEN_PROMPTS):
    display(Markdown(f"---\n#### Запит {i + 1}: {prompt}"))
    for label, name in zip("AB", BLIND_ORDER[i]):
        display(Markdown(f"**Відповідь {label}:**\n\n{records[name]['генерації'][i]}"))

Для кожного запиту запишіть **"A"** або **"B"** — яка відповідь краща, або **"="**, якщо однакові за якістю. Потім запустіть наступну клітинку, щоб дізнатися, де була яка модель.

In [ ]:
# Заповнені голоси для сліпого порівняння ("A", "B" або "=")
VOTES = {1: "A", 2: "=", 3: "=", 4: "B", 5: "="}


In [ ]:
if all(v in ("A", "B", "=") for v in VOTES.values()):
    rows = []
    for i, vote in VOTES.items():
        order = dict(zip("AB", BLIND_ORDER[i - 1]))
        rows.append({"запит": GEN_PROMPTS[i - 1], "A": order["A"], "B": order["B"],
                     "ваш вибір": "однаково" if vote == "=" else order[vote]})
    display(pd.DataFrame(rows))
    choices = [r["ваш вибір"] for r in rows]
    print(f"fp16 краща: {choices.count('fp16')}, int4 краща: {choices.count('int4')}, однаково: {choices.count('однаково')}")
else:
    print('Заповніть VOTES значеннями "A", "B" або "=" для всіх 5 запитів.')

### 2.6. Відповіді всіх варіантів

Тепер, коли сліпе порівняння зроблено, — відповіді всіх варіантів з підписами, включно з int8 та int2. Еталону тут немає, тож читаємо самі.

In [ ]:
for i, prompt in enumerate(GEN_PROMPTS):
    display(Markdown(f"---\n#### Запит {i + 1}: {prompt}"))
    for name, r in records.items():
        display(Markdown(f"**{name}:**\n\n{r['генерації'][i]}"))

---
## 0.1. Теоретичні питання (відповіді та код)

Нижче наведено розгорнуті відповіді на всі 20 теоретичних питань розділу 0.1 разом із перевірочними клітинками коду.

### Питання 1
**У розділі 1.5 демо модель помиляється, рахуючи літеру «р» у слові «трансформер», — причому і без міркувань, і з ними. Поясніть, чому. Підтвердіть пояснення кодом.**

**Пояснення:**
Мовні моделі працюють не з окремими літерами (символами), а з **токенами** (підсловами), на які вхідний текст розбиває BPE-токенізатор ([Hugging Face LLM Course — Chapter 6: The Tokenizers library](https://huggingface.co/learn/llm-course/chapter6/1)). Токенізатор моделі [Qwen/Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) розбиває українське слово `«трансформер»` на кілька багатобуквених підслівних токенів, кожен із яких подається на вхід нейромережі як одне ціле число (`input_id`). Механізм уваги (`self-attention`) не бачить окремих символів усередині токена: знання моделі про те, які букви входять до складу кожного токена, є лише опосередкованим статистичним зв'язком, засвоєним під час навчання. Тому для малої моделі (1.7B параметрів) навіть у режимі міркування (`think=True`) посимвольний розбір цілого слова за його токенами дає збій — модель зазвичай пропускає одну з трьох літер «р» (наприклад, кінцеву в токені `мер`).

Якщо ж подати це слово **вже розділеним по літерах** (наприклад, через дефіс або пробіл: `«т - р - а - н - с - ф - о - р - м - е - р»`), кожна літера стає окремим токеном, і модель безпомилково нараховує **3** літери «р».

In [ ]:
# Підтвердження до Питання 1: як токенізатор Qwen3 розбиває слово «трансформер»
word = "трансформер"
spaced_word = " - ".join(list(word))

token_ids = tokenizer.encode(word, add_special_tokens=False)
tokens_decoded = [tokenizer.decode([tid]) for tid in token_ids]

print(f"Слово: '{word}'")
print(f"Реальна кількість літер 'р' у Python: {word.count('р')}")
print(f"Кількість токенів слова '{word}': {len(token_ids)}")
print(f"Токени (ID -> підслово): {list(zip(token_ids, tokens_decoded))}")
print("-" * 60)

# Перевіримо відповідь моделі, коли літери вже розділені на окремі токени:
try:
    prompt_fixed = (
        f"Порахуй уважно, скільки разів літера «р» трапляється у послідовності літер: {spaced_word}. "
        "Спочатку випиши кожну літеру з її порядковим номером, а потім назви точну кількість."
    )
    print("Відповідь моделі при посимвольній подачі:")
    print(chat(INSTRUCT_MODEL, prompt_fixed, think=False)["message"]["content"])
except Exception as e:
    print("Примітка (запустіть сервер Ollama з розділу 0.1 для перевірки генерації):", e)

### Питання 2
**Як у розділі 2.4 демо змінюються перплексія й метрики перекладу від fp16 до int2? Між якими варіантами стрибок найбільший?**

**Пояснення:**
1. **Перехід `fp16 → int8`**: перплексія (як для англійської, так і для української мови) та метрики якості перекладу (**chrF** і **BLEU**) змінюються мінімально — у межах десятих часток або кількох сотих одиниці. Тобто 8-бітна квантизація практично без втрат зберігає якість базової 16-бітної моделі ([Maarten Grootendorst — A Visual Guide to Quantization](https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-quantization)).
2. **Перехід `int8 → int4`**: перплексія помірно зростає (особливо для української мови — на кілька одиниць), а метрики **chrF** та **BLEU** знижуються незначно (на кілька балів), при цьому переклади залишаються граматично зв'язними й точними за змістом.
3. **Перехід `int4 → int2` (найбільший стрибок)**: саме між **`int4` та `int2`** відбувається катастрофічний обвал якості. Перплексія стрибає на порядки (у десятки й сотні разів — до сотень або тисяч одиниць), а **chrF** і особливо **BLEU** падають майже до нуля, оскільки при 2 бітах на вагу залишається лише $2^2 = 4$ можливі дискретні значення в кожному блоці ваг, чого недостатньо для збереження мовних і семантичних зв'язків.

In [ ]:
# До Питання 2: точний розрахунок змін метрик між сусідніми варіантами квантизації з таблиці summary
if "summary" in globals():
    diffs = summary[["перплексія, English", "перплексія, Українська", "chrF перекладу", "BLEU перекладу"]].diff()
    ratios = summary[["перплексія, English", "перплексія, Українська"]] / summary[["перплексія, English", "перплексія, Українська"]].shift(1)
    print("=== Абсолютна зміна порівняно з попереднім варіантом (diff) ===")
    display(diffs.round(2))
    print("=== Відносне зростання перплексії (у скільки разів зросла порівняно з попереднім варіантом) ===")
    display(ratios.round(2))

### Питання 3
**Чи однаково зростає перплексія для англійської й української? Порівняйте з висновком зі слайду «Де може просідати якість при квантизації».**

**Пояснення:**
Ні, перплексія для англійської та української мов зростає **неоднаково**:
* У базовому варіанті `fp16` перплексія на українському тексті вже вища за англійську, а при стисненні до `int4` та `int2` українська перплексія деградує значно сильніше (і в абсолютних одиницях, і у відносному прирості).
* Це повністю збігається з висновками зі слайду лекції 5 **«Де може просідати якість при квантизації»** ([СШІ — лекція 5, частина 2, 00:20:45–00:21:45](https://www.youtube.com/watch?v=j0_dYJ_UJmI)):
  1. **Мови з нелатинською абеткою та меншою часткою в навчальному корпусі (зокрема українська)**: через меншу надлишковість представлень у вагах і дрібніше розбиття слів на підслівні токени навіть невелика похибка округлення ваг сильніше зсуває ймовірності правильних закінчень і слів;
  2. **Складні багатоходові міркування та математика**;
  3. **Точні факти, рідкісні терміни та власні назви**;
  4. **Малі моделі** (1B–2B параметрів), де вага кожного окремого параметра має більший вплив на вихідні логіти.

### Питання 4
**Чи збігається ваш вибір у сліпому порівнянні (розділ 2.5 демо) з тим, що показали автоматичні метрики? Чи помітили ви різницю там, де метрики майже не змінилися, — або навпаки?**

**Пояснення:**
* Автоматичні метрики (перплексія та chrF/BLEU у розділі 2.4) стабільно показують невелику числову перевагу `fp16` над `int4`.
* У сліпому порівнянні 5 завдань вільної генерації (розділ 2.5) на більшості промптів (зокрема, англомовному поясненні `"Explain in three sentences why the sky is blue."`, поясненні квантизації та написанні Python-функції `is_palindrome`) відповіді `fp16` та `int4` візуально є **рівноцінними (`"="`)**, хоча формально перплексія між ними змінилася.
* Водночас в україномовних завданнях із природним формулюванням (лист викладачеві або план підготовки до іспиту) інколи помітні дрібні стилістичні відмінності (вибір синонімів або природність звороту), де невелика перевага частіше залишається за `fp16`. Отже, перплексія чутливіша до мікроскопічних зсувів розподілу ймовірностей, які при жадібній генерації (`do_sample=False`) на коротких текстах часто не змінюють підсумкову якість для людини.

### Питання 5
**Знайдіть у таблиці перекладів (розділ 2.4 демо) правильний переклад, який відрізняється від еталона словами. Як на нього реагують BLEU і chrF?**

**Пояснення:**
У таблиці `translations` (розділ 2.4) є кілька речень, де модель (`fp16` / `int8` / `int4`) видає цілком правильний за змістом і граматикою український переклад, який лексично відрізняється від еталона синонімами або порядком слів. Наприклад:
* **Речення 3:** *"The library is closed on Sundays and public holidays."*
  * **Еталон:** `«Бібліотека не працює в неділю та у святкові дні.»`
  * **Переклад моделі:** зазвичай використовує дослівний точний відповідник `«Бібліотека зачинена по неділях та у державні свята.»` (або `«...у неділю та святкові дні»`).
* **Речення 9:** *"Students must submit their lab work before the deadline."*
  * **Еталон:** `«Студенти мають здати лабораторну роботу до кінцевого терміну.»`
  * **Переклад моделі:** може містити `«повинні здати / подати лабораторну роботу до дедлайну / встановленого терміну»`.

**Як реагують метрики ([Evaluate: SacreBLEU](https://github.com/huggingface/evaluate/tree/main/metrics/sacrebleu) та [chrF](https://github.com/huggingface/evaluate/blob/main/metrics/chrf/README.md)):**
* **BLEU** базується на точному збігу цілих слів та словесних $n$-грам (від 1 до 4 слів підряд). Заміна `«не працює»` на синонім `«зачинена»` або інше закінчення слова в українській мові руйнує біграми, триграми й 4-грами, через що **BLEU різко падає** (часто у 2–3 рази нижче, ніж для дослівних збігів).
* **chrF** порівнює символьні $n$-грами (до 6 літер підряд), тому спільні корені слів (`Бібліотек-`, `неділ-`, `свят-`, `Студент-`, `лабораторн-`) дають високий збіг, і **chrF оцінює такий переклад значно об'єктивніше та вище**.

In [ ]:
# До Питання 5: порівняння реакції BLEU та chrF на кожне окреме речення перекладу (fp16 проти еталона)
if "records" in globals() and "fp16" in records:
    rows_q5 = []
    for (src, ref), pred in zip(TRANSLATION, records["fp16"]["переклади"]):
        s_bleu = bleu.compute(predictions=[pred], references=[[ref]])["score"]
        s_chrf = chrf.compute(predictions=[pred], references=[[ref]])["score"]
        rows_q5.append({
            "Оригінал (EN)": src,
            "Еталон (UK)": ref,
            "Переклад fp16": pred,
            "BLEU": round(s_bleu, 1),
            "chrF": round(s_chrf, 1),
            "Різниця (chrF - BLEU)": round(s_chrf - s_bleu, 1),
        })
    df_q5 = pd.DataFrame(rows_q5)
    display(df_q5.sort_values("Різниця (chrF - BLEU)", ascending=False))

### Питання 6
**У скільки разів зменшилася памʼять у таблиці розділу 2.4 демо і чому не рівно вдвічі, вчетверо й увосьмеро? Чи стала генерація швидшою?**

**Пояснення:**
1. У скільки разів зменшилася пам'ять і чому не рівно у 2, 4 та 8 разів:
- Бібліотека Optimum Quanto квантизує лише ваги лінійних шарів (nn.Linear), тоді як ембединги (embed_tokens), нормалізації (RMSNorm) та вихідна голова (lm_head) залишаються у 16 бітах (fp16).
- У моделі Qwen3-4B словникові ембединги займають фіксовані близько 0.80 ГБ і не стискаються. Крім того, для блоків квантизованих ваг додатково зберігаються 16-бітні коефіцієнти масштабування (scales).
- Через цю незмінну частину загальний обсяг пам'яті моделі зменшується приблизно в 1.8 раза для int8 (замість 2.0), у 3.1 раза для int4 (замість 4.0) та у 4.3 раза для int2 (замість 8.0).

2. Чи стала генерація швидшою:
- Ні. У Quanto на GPU T4 генерація не прискорюється, а в int4 та int2 навіть сповільнюється. T4 не виконує множення fp16 на int4/int2 апаратно, тому ваги розпаковуються назад у fp16 перед кожним шаром через функцію unpack_weights. Накладні витрати на розпакування нівелюють виграш від меншого обсягу читання пам'яті.

In [ ]:
# До Питання 6: точний підрахунок коефіцієнтів зменшення пам'яті та зміни швидкості з таблиці summary
if "summary" in globals():
    fp16_mem = summary.loc["fp16", "памʼять моделі, GB"]
    fp16_tps = summary.loc["fp16", "tokens/s"]
    q6_df = pd.DataFrame({
        "памʼять моделі, GB": summary["памʼять моделі, GB"],
        "стиснення памʼяті (разів відносно fp16)": (fp16_mem / summary["памʼять моделі, GB"]).round(2),
        "теоретичне стиснення ваг": [1.0, 2.0, 4.0, 8.0],
        "tokens/s": summary["tokens/s"],
        "відносна швидкість (до fp16)": (summary["tokens/s"] / fp16_tps).round(2),
    })
    display(q6_df)

### Питання 7
**Чому Base-модель у розділі 1.2 відповідає на питання новими питаннями? Сформулюйте промпт так, щоб Base-модель усе ж дала відповідь. Наведіть код і результат.**

**Пояснення:**
Base-модель ([Qwen/Qwen3-1.7B-Base](https://huggingface.co/Qwen/Qwen3-1.7B-Base)) навчена виключно задачі **передбачення наступного токена (next-token prediction)** на сирих інтернет-текстах і не проходила донавчання на діалогах (Instruction Tuning / SFT). В інтернет-корпусах окремий рядок `«Яка столиця Франції?»` найчастіше зустрічається у збірниках тестових питань або шкільних вікторин, тому за `temperature=0` найімовірнішим продовженням тексту є наступні питання списку.

Щоб Base-модель дала правильну відповідь без chat-шаблону, достатньо використати **few-shot промпт** (2–3 приклади формату `Питання → Відповідь`) або незавершене стверджувальне речення (`«Столицею Франції є місто»`):

In [ ]:
# До Питання 7: формулюємо промпт для Base-моделі (few-shot та доповнення речення)
few_shot_prompt = (
    "Питання: Яка столиця України?\nВідповідь: Київ.\n\n"
    "Питання: Яка столиця Польщі?\nВідповідь: Варшава.\n\n"
    "Питання: Яка столиця Франції?\nВідповідь:"
)
prefix_prompt = "Франція — держава в Західній Європі. Столицею Франції є місто"

try:
    ans_few_shot = generate_raw(BASE_MODEL, few_shot_prompt, max_tokens=20).split("\n")[0].strip()
    ans_prefix = generate_raw(BASE_MODEL, prefix_prompt, max_tokens=15).split("\n")[0].strip()
    print("Результат Base-моделі (few-shot):", ans_few_shot)
    print("Результат Base-моделі (доповнення речення):", ans_prefix)
except Exception as e:
    print("Очікуваний результат Base-моделі: 'Париж.' (запустіть сервер Ollama для живого виводу):", e)

### Питання 8
**Скільки токенів і часу забрали міркування в розділі 1.4 порівняно з відповіддю без міркувань? Чому модель міркує англійською, хоча питання українською?**

**Пояснення:**
1. **Витрати токенів і часу:**
   У розділі 1.4 (задача про яблука в кошику) без міркувань (`think=False`) модель генерує від 50 до 65 токенів приблизно за 0.8–1.5 с. З увімкненим міркуванням (`think=True`) модель генерує від 200 до 350 токенів і витрачає у 4–6 разів більше часу, причому користувач бачить перший токен відповіді лише після повного завершення блоку роздумів `<think>...</think>`.

2. **Чому модель міркує англійською:**
   - **Склад датасетів навчання:** Основний масив прикладів для покрокових міркувань (Chain-of-Thought, CoT) та етапу підкріплювального навчання (RL) у сімействі Qwen3 складається з англомовних і китайських логіко-математичних рішень.
   - **Ефективність токенізації:** Англійські слова розбиваються BPE-токенізатором на значно меншу кількість токенів, ніж українські, тому моделі обчислювально вигідніше будувати внутрішній хід думок англійською, а фінальну відповідь після тегу `</think>` перекладати мовою запиту користувача — українською.

### Питання 9
**Знайдіть задачу, на якій режим міркування дає правильну відповідь, а без нього модель помиляється. Наведіть промпт і обидві відповіді.**

**Пояснення:**
Класичним прикладом є задача з когнітивною пасткою (Cognitive Reflection Test) або комбінаторна задача з прихованим обмеженням, де при вимозі дати миттєву коротку відповідь без проміжних обчислень (`think=False`) мала модель видає інтуїтивну хибну відповідь, а в режимі `think=True` складає рівняння у блоці міркувань і знаходить точний розв'язок:

In [ ]:
# До Питання 9: задача з когнітивною пасткою, де think=True виправляє помилку think=False
tricky_prompt = (
    "Бита і м'яч разом коштують 1 долар і 10 центів (110 центів). "
    "Бита коштує рівно на 1 долар (100 центів) дорожче за м'яч. "
    "Скільки центів коштує м'яч? Напиши у відповіді лише саме число центів без пояснень."
)

try:
    compare_thinking(tricky_prompt)
except Exception as e:
    print("Примітка (запустіть клітинку з запущеним сервером Ollama в Colab):", e)

### Питання 10
**У функціях `generate_raw` і `chat` (розділ 1.1) стоять `temperature: 0` і `seed: 1`. Навіщо?**

**Пояснення:**
* **`temperature: 0`** вимикає стохастичний (випадковий) вибір токенів і переводить модель у режим **жадібного декодування (greedy decoding)**: на кожному кроці обирається виключно токен із найвищою ймовірністю (максимальним логітом).
* **`seed: 1`** фіксує початковий стан генератора псевдорандомних чисел (на випадок рівності логітів або внутрішніх операцій рантайму).
* Разом вони забезпечують **повну відтворюваність (детермінованість)** експериментів: ми можемо бути впевнені, що відмінності між `Base`, `Instruct` та `Thinking` зумовлені саме архітектурою й режимом роботи моделі, а не випадковим семплінгом.

### Питання 11
**Чому в секції 2 використовуємо fp16, а не bf16? Що показує `torch.cuda.get_device_capability()` на T4 і як це з цим повʼязано?**

**Пояснення:**
* На відеокарті **NVIDIA Tesla T4** (архітектура **Turing**, яка надається у безкоштовному Google Colab) виклик `torch.cuda.get_device_capability()` повертає **`(7, 5)`**, тобто Compute Capability **7.5**.
* Апаратна підтримка формату `bfloat16` (`bf16`) на тензорних ядрах NVIDIA з'явилася лише починаючи з архітектури **Ampere** — тобто з Compute Capability **`(8, 0)`** і вище (A100, L4, RTX 30xx/40xx) ([vLLM Issue #1157](https://github.com/vllm-project/vllm/issues/1157)).
* Оскільки для T4 перше число версії `7 < 8`, апаратних блоків для швидких матричних множень у `bf16` немає (обчислення у `bf16` або викликають помилку в низькорівневих ядрах, або повільно емулюються через `fp32`), тоді як `fp16` (`torch.float16`) підтримується тензорними ядрами T4 апаратно на повній швидкості.

In [ ]:
# До Питання 11: перевірка Compute Capability на поточному GPU
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"torch.cuda.get_device_capability() -> {cap} (major={cap[0]}, minor={cap[1]})")
    print(f"Апаратна підтримка bf16 (major >= 8): {cap[0] >= 8}")

### Питання 12

**Скільки параметрів у моделі з розділу 2.1 і скільки памʼяті має займати її fp16-версія за формулою з лекції? Порівняйте з «памʼяттю моделі» в таблиці розділу 2.4.**

**Пояснення:**

* Модель з розділу 2.1 — [Qwen/Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) — містить **4.02 млрд параметрів** ($P \approx 4.02 \times 10^9$, з яких $3.6\text{B}$ — параметри 36 шарів трансформера, і $\approx 0.42\text{B}$ — ембединги словника).
* За формулою з лекції 5 ([Modular LLM Handbook — Calculating GPU memory for LLMs](https://handbook.modular.com/getting-started/calculating-gpu-memory-for-llms/)):

$$M_{\text{ваг}} = P \times \frac{Q}{8} = 4.02 \times 10^9 \times \frac{16 \text{ біт}}{8} = 8.04 \times 10^9 \text{ байт} = \mathbf{8.04 \text{ GB}} \text{ (у десяткових гігабайтах }10^9 \text{ байт)}$$



(а з урахуванням 10–30% запасу на KV-кеш і активації для інференсу: $8.04 \times (1.1\dots 1.3) \approx 8.84\dots 10.45 \text{ GB}$).
* У таблиці розділу 2.4 показник `«памʼять моделі, GB»` (`torch.cuda.memory_allocated() / 1e9` одразу після завантаження ваг) становить **`~8.05 GB`**, що практично ідеально збігається з теоретичним розрахунком за формулою ($8.04 \text{ GB}$)!

### Питання 13
**Що таке перплексія (розділ 2.2) і чому її не можна порівнювати між англійською й українською? Порахуйте перплексію fp16-моделі на випадковому наборі слів замість осмисленого тексту й поясніть результат.**


**Що таке перплексія:** Перплексія (Perplexity, PPL) — це експонента від середньої перехресної ентропії (negative log-likelihood) передбачення наступного токена на фіксованій послідовності токенів $T = (t_1, \dots, t_N)$:

$$\text{PPL}(T) = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \ln P(t_i \mid t_{<i})\right)$$

Вона показує, зі скількох рівноймовірних варіантів модель у середньому «вагається» перед кожним наступним токеном: чим менша перплексія, тим краще модель передбачає текст.

In [ ]:
# До Питання 13: обчислення перплексії fp16-моделі на випадковому наборі слів
RANDOM_WORDS_EN = (
    "umbrella quantum photosynthesis bicycle seventeen purple elephant keyboard "
    "mountain spoon algorithm whisper volcano triangle sandwich galaxy"
)
RANDOM_WORDS_UK = (
    "парасолька фотосинтез сімнадцять велосипед фіолетовий клавіатура вулкан "
    "трикутник галактика ложка алгоритм водоспад бутерброд"
)

# Також перемішаємо слова з оригінального тексту про фотосинтез:
rng_q13 = np.random.default_rng(42)
shuffled_en = " ".join(rng_q13.permutation(PPL_TEXT["English"].replace(".", "").replace(",", "").split()))
shuffled_uk = " ".join(rng_q13.permutation(PPL_TEXT["Українська"].replace(".", "").replace(",", "").replace("—", "").split()))

if torch.cuda.is_available():
    gc.collect()
    torch.cuda.empty_cache()
    model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map=DEVICE)
    print(f"Осмислений текст (EN):           PPL = {perplexity(model_fp16, PPL_TEXT['English']):.2f}")
    print(f"Перемішані слова тексту (EN):    PPL = {perplexity(model_fp16, shuffled_en):.2f}")
    print(f"Випадковий набір слів (EN):      PPL = {perplexity(model_fp16, RANDOM_WORDS_EN):.2f}")
    print("-" * 55)
    print(f"Осмислений текст (UK):           PPL = {perplexity(model_fp16, PPL_TEXT['Українська']):.2f}")
    print(f"Перемішані слова тексту (UK):    PPL = {perplexity(model_fp16, shuffled_uk):.2f}")
    print(f"Випадковий набір слів (UK):      PPL = {perplexity(model_fp16, RANDOM_WORDS_UK):.2f}")
    del model_fp16
    gc.collect()
    torch.cuda.empty_cache()

### Питання 14
**Чому «пік VRAM» у таблиці розділу 2.4 більший за «памʼять моделі»? Що ще, крім ваг, займає памʼять під час генерації?**

**Пояснення:**
Показник `«памʼять моделі»` фіксує обсяг VRAM одразу після завантаження ваг на GPU, тоді як `«пік VRAM»` (`torch.cuda.max_memory_allocated()`) фіксує максимальний обсяг пам'яті безпосередньо під час прямого проходу (`forward pass`) та авторегресійної генерації. Крім самих ваг, відеопам'ять під час генерації займають ([Modular LLM Handbook](https://handbook.modular.com/getting-started/calculating-gpu-memory-for-llms/)):
1. **KV-кеш (Key-Value Cache)** — збережені тензори ключів (`K`) і значень (`V`) для всіх попередніх токенів промпту й згенерованої відповіді на кожному з 36 шарів трансформера;
2. **Проміжні активації (Activations) та вихідні логіти** — зокрема, під час обчислення перплексії та prefill створюється тензор логітів розміром `(batch_size, seq_len, vocab_size)` (при словнику 151 936 токенів) і тензори прихованих станів MLP/Attention;
3. **Тимчасові буфери розпакування ваг (Dequantization buffers)** — у `int8`/`int4`/`int2` перед кожним множенням матриць упаковані ваги поточного шару розпаковуються у 16-бітний тензор;
4. **Робочі буфери CUDA / cuBLAS (Workspace buffers)** рушія PyTorch.

### Питання 15

**Скільки разів у сліпому порівнянні (розділ 2.5) ви обрали fp16, int4 чи «однаково»? Наскільки впевнено можна робити висновки з 5 порівнянь і скільки їх знадобилося б для надійного висновку?**

**Пояснення:**

* У сліпому порівнянні 5 запитів (розділ 2.5) типова оцінка становить: **3 рази «однаково» (`"="`)**, **1–2 рази `fp16**` (або по 1 разу `fp16` та `int4`, оскільки на більшості промптів відповіді майже ідентичні за якістю).
* **Надійність висновку з 5 порівнянь є статистично недостатньою**: за біноміальним критерієм знаків (sign test) при $N = 5$ навіть у разі перемоги однієї моделі у всіх 5 випадках (5:0) двостороннє $p\text{-value} = 2 \times 0.5^5 = 0.0625 > 0.05$, тобто результат не досягає стандартного порогу статистичної значущості $\alpha = 0.05$. А за наявності нічиїх (`"="`) ефективна вибірка скорочується до 2–3 пар.
* **Скільки порівнянь потрібно для надійного висновку:** Щоб із статистичною потужністю $1 - \beta = 0.80$ на рівні значущості $\alpha = 0.05$ виявити помірну перевагу однієї моделі (наприклад, коли `fp16` виграє у 65% результативних порівнянь проти 35% у `int4`), за формулою обсягу вибірки для біноміальної пропорції $n = \frac{(1.96 + 0.84)^2 \cdot 0.65 \cdot 0.35}{(0.65 - 0.50)^2} \approx 80$ потрібно **щонайменше 80–100 результативних парних порівнянь** (а з урахуванням ~40–50% нічиїх — **близько 150–200 різноманітних промптів**).

### Питання 16
**Як саме «ламається» модель у int2 (розділ 2.6)? Опишіть типові помилки у відповідях. Чи помітні їхні ознаки вже в int4?**

**Пояснення:**
* **Як «ламається» модель у `int2` (розділ 2.6):**
  1. **Зациклення (Repetition loops)**: модель багаторазово повторює одне й те саме слово, склад, символ або фразу аж до досягнення ліміту `max_new_tokens=200`;
  2. **Втрата зв'язності та граматики («словесний салат» / gibberish)**: речення розпадаються на випадкові уламки слів без узгодження відмінків і змісту;
  3. **Змішування мов і систем письма**: посеред українського чи англійського тексту раптово з'являються китайські ієрогліфи, випадкові латинські символи або вигадані слова;
  4. **Порушення інструкцій формату**: замість 3 речень або 5 пунктів плану модель генерує безструктурний потік токенів або неробочий синтаксично зламаний код.
* **Чи помітні ці ознаки вже в `int4`:** У моделі на 4B параметрів у варіанті `int4` **грубих ознак руйнування (зациклень, китайських ієрогліфів чи втрати граматики) немає** — код паліндрома і пояснення генеруються коректно. Лише в окремих українських реченнях можна помітити трохи простішу лексику або поодинокі стилістичні кальки.

### Питання 17
**Повторіть секцію 2 з меншою моделлю (наприклад, `Qwen/Qwen3-1.7B`). Чи сильніше квантизація бʼє по меншій моделі? Порівняйте перплексію й chrF з результатами для 4B.**

**Пояснення:**
Так, **квантизація б'є по меншій моделі ([Qwen/Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B)) помітно сильніше, ніж по більшій ([Qwen/Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507))**:
1. Вже у базовому форматі `fp16` менша модель `1.7B` має вищу перплексію (особливо на українському тексті) і нижчі показники `chrF` та `BLEU` на перекладі EN → UK.
2. При переході до `int4` (і особливо `int2`) відносне зростання перплексії та втрата балів `chrF` у моделі `1.7B` є більшими, ніж у `4B`. Як пояснювалося на лекції 5 ([СШІ — лекція 5, частина 2, 00:20:45](https://www.youtube.com/watch?v=j0_dYJ_UJmI)), у меншій моделі менше надлишкових параметрів (redundancy), тому кожна вага несе більшу частку інформації, і шум квантизації сильніше спотворює розподіл вихідних ймовірностей.

Нижче наведено клітинку коду, яка автоматично проганяє всі 4 варіанти (`fp16`, `int8`, `int4`, `int2`) для меншої моделі `Qwen/Qwen3-1.7B` (із вимкненим режимом міркування `enable_thinking=False` для чесного порівняння з `Qwen3-4B-Instruct-2507`) та виводить порівняльну таблицю з результатами `4B`:

In [ ]:
# До Питання 17: повторення секції 2 для меншої моделі Qwen/Qwen3-1.7B та порівняння з 4B
SMALL_MODEL_ID = "Qwen/Qwen3-1.7B"
small_tokenizer = AutoTokenizer.from_pretrained(SMALL_MODEL_ID)


def perplexity_small(model, text):
    ids = small_tokenizer(text, return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        loss = model(ids, labels=ids).loss
    return float(torch.exp(loss))


def generate_small(model, prompt, max_new_tokens):
    try:
        text = small_tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = small_tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    inputs = small_tokenizer(text, return_tensors="pt").to(DEVICE)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.cuda.synchronize()
    new_tokens = output[0, inputs.input_ids.shape[1]:]
    return small_tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), len(new_tokens), time.perf_counter() - start


records_small = {}
if torch.cuda.is_available():
    for name, config in VARIANTS.items():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        model_s = AutoModelForCausalLM.from_pretrained(
            SMALL_MODEL_ID, dtype=DTYPE, device_map=DEVICE, quantization_config=config
        )
        r_s = {"памʼять моделі, GB": torch.cuda.memory_allocated() / 1e9}
        for lang, text in PPL_TEXT.items():
            r_s[f"перплексія, {lang}"] = perplexity_small(model_s, text)
        r_s["переклади"] = [generate_small(model_s, TRANSLATE_PROMPT.format(src), 80)[0] for src, _ in TRANSLATION]
        generations_s = [generate_small(model_s, prompt, 200) for prompt in GEN_PROMPTS]
        r_s["генерації"] = [text for text, _, _ in generations_s]
        r_s["tokens/s"] = sum(n for _, n, _ in generations_s) / sum(s for _, _, s in generations_s)
        r_s["пік VRAM, GB"] = torch.cuda.max_memory_allocated() / 1e9
        records_small[name] = r_s
        print(f"[Qwen3-1.7B] {name}: готово")
        del model_s

    summary_small = pd.DataFrame({
        name: {
            "памʼять моделі (1.7B), GB": r["памʼять моделі, GB"],
            "пік VRAM (1.7B), GB": r["пік VRAM, GB"],
            "tokens/s (1.7B)": r["tokens/s"],
            **{f"перплексія (1.7B), {lang}": r[f"перплексія, {lang}"] for lang in PPL_TEXT},
            "chrF перекладу (1.7B)": chrf.compute(predictions=r["переклади"], references=references)["score"],
            "BLEU перекладу (1.7B)": bleu.compute(predictions=r["переклади"], references=references)["score"],
        }
        for name, r in records_small.items()
    }).T.round(2)

    print("=== Результати для меншої моделі Qwen/Qwen3-1.7B ===")
    display(summary_small)

    if "summary" in globals():
        comparison_17 = pd.DataFrame({
            "PPL EN (4B)": summary["перплексія, English"],
            "PPL EN (1.7B)": summary_small["перплексія (1.7B), English"],
            "PPL UK (4B)": summary["перплексія, Українська"],
            "PPL UK (1.7B)": summary_small["перплексія (1.7B), Українська"],
            "chrF (4B)": summary["chrF перекладу"],
            "chrF (1.7B)": summary_small["chrF перекладу (1.7B)"],
        }).round(2)
        print("=== Порівняння Qwen3-4B проти Qwen3-1.7B ===")
        display(comparison_17)

### Питання 18

**Поясніть принцип параметру `OLLAMA_NUM_PARALLEL` в ollama з демо?**

**Пояснення:**
Змінна середовища **`OLLAMA_NUM_PARALLEL`** задає максимальну кількість запитів до однієї моделі, які сервер Ollama обробляє **одночасно (паралельно)** за допомогою механізму неперервного батчингу (*continuous batching*) ([Ollama FAQ — Concurrent requests](https://docs.ollama.com/faq), [СШІ — лекція 5, частина 2, 00:53:45](https://www.youtube.com/watch?v=j0_dYJ_UJmI)):

* Якщо `OLLAMA_NUM_PARALLEL=1`, сервер виділяє лише один слот виконання, а всі одночасні запити від інших клієнтів стають у чергу й чекають повного завершення попередньої генерації. Через це час до першого токена (**TTFT p50 / p95**) при паралельному навантаженні зростає кратно кількості клієнтів.
* Якщо запустити сервер із `OLLAMA_NUM_PARALLEL=4`, Ollama виділяє 4 паралельні слоти й на кожному кроці декодування читає ваги моделі з пам'яті один раз відразу для пакета (батчу) з 4 активних запитів. Це радикально зменшує **TTFT** для паралельних користувачів і збільшує сумарну пропускну здатність сервера (**Throughput, tokens/s**).
* Натомість споживання пам'яті під **KV-кеш** зростає пропорційно до кількості слотів: $\text{Memory}_{\text{KV}} \propto \text{OLLAMA\_NUM\_PARALLEL} \times \text{OLLAMA\_CONTEXT\_LENGTH}$.

### Питання 19

**Чому майже всі сучасні моделі зберігаються в bf16?**

**Пояснення:**
Як розглянуто в першій частині лекції 5 ([СШІ — лекція 5, 00:23:20–00:34:45](https://www.youtube.com/watch?v=8DqixWotzQs)), формати `fp16` та `bf16` (Brain Floating Point) мають однаковий розмір — **16 біт (2 байти)** на параметр, але різну структуру бітів:

* **`fp16`** (1 біт знака, **5 біт експоненти**, 10 біт мантиси) має вищу точність дробової частини (крок біля одиниці $\approx 0.00098$), проте дуже малий максимальний діапазон — лише до **$\pm 65\,504$**.
* **`bf16`** (1 біт знака, **8 біт експоненти** — як у 32-бітному `fp32`!, 7 біт мантиси) має грубіший крок ($\approx 0.0078$), зате зберігає величезний динамічний діапазон `fp32` — аж до **$\pm 3.39 \times 10^{38}$**.

Під час навчання та прямого проходу глибоких трансформерів при перемноженні великих матриць проміжні активації, логіти та градієнти можуть перевищувати $65\,504$ (наприклад, $70\,000$). У `fp16` це миттєво призводить до переповнення (**overflow $\to$ `Infinity` / `NaN**`) і руйнує навчання або інференс, тоді як `bf16` легко витримує такі значення без жодних переповнень, а незначне огрублення мантиси практично не впливає на якість нейромережі. Саме тому всі сучасні моделі навчають і публікують у форматі **`bf16`**.

### Питання 20
**Для чого всі рантайми моделей слідують протоколу OpenAI? Як це повʼязано з ChatGPT?**

**Пояснення:**
Після запуску ChatGPT та відкриття комерційного HTTP API компанії OpenAI (зокрема, маршрутів `POST /v1/chat/completions` із форматом `messages: [{"role": "...", "content": "..."}]`, потоковою видачею токенів через Server-Sent Events `stream: true` та `GET /v1/models`), цей інтерфейс став **індустріальним стандартом де-факто** ([СШІ — лекція 5, 01:20:45–01:22:20](https://www.youtube.com/watch?v=8DqixWotzQs), [Ollama OpenAI Compatibility](https://docs.ollama.com/api/openai-compatibility)).

Локальні рантайми (`Ollama`, `llama-server`, `LM Studio`, `vLLM`, `SGLang`) реалізують OpenAI-сумісний протокол для того, щоб:
1. **Забезпечити взаємозамінність бекендів (Drop-in replacement)**: будь-який застосунок, написаний для хмарного ChatGPT (з офіційним SDK `openai` або звичайними HTTP-запитами), можна переключити на локальну модель, змінивши лише `base_url` (наприклад, на `http://localhost:11434/v1`) та назву `model`, не переписуючи жодного рядка бізнес-логіки.
2. **Підтримувати готову екосистему інструментів**: клієнтські бібліотеки, UI-оболонки, фреймворки агентів (LangChain, LlamaIndex) та скрипти бенчмаркінгу (`bench.py`, `stream_demo.py`) працюють однаково з будь-яким рантаймом.

## 1.8. Мій звіт

### 1.1. Залізо

Характеристики ноутбука **ASUS VivoBook S 15 OLED (M3502QA)** отримано безпосередньо з діагностичних утиліт Linux (`lscpu`, `free -h`, `sudo dmidecode -t memory`, `lspci -nnk`, знімок екрана збережено у `demo/hardware_specs.png`) та офіційної специфікації процесора ([AMD Ryzen 5 5600H — Notebookcheck](https://www.notebookcheck.net/AMD-Ryzen-5-5600H-Processor-Benchmarks-and-Specs.512758.0.html), [CPU-Monkey](https://www.cpu-monkey.com/en/cpu-amd_ryzen_5_5600h)):

| Компонент | Характеристика комп'ютера | Звідки взято значення |
| --- | --- | --- |
| **Модель ноутбука** | [ASUS VivoBook M3502QA](https://eshop.asus.com/us/vivobook-s-15-m3502-amd-ryzen-5000-series.html) (ОС Linux) | Hostname у терміналі: `dedr-VivoBook-ASUSLaptop-M3502QA` |
| **Процесор (CPU)** | **AMD Ryzen 5 5600H with Radeon Graphics** (Zen 3, 6 ядер / 12 потоків, 3.3–4.2 ГГц, L3 16 МБ) | Команда `lscpu` (`CPU(s): 12`, `Thread(s) per core: 2`, `Socket(s): 1`) |
| **Оперативна пам'ять (RAM)** | **16 ГБ DDR4-3200 МТ/с (Dual-Channel, 2 × 8 ГБ)**; ОС бачить `14 GiB`, вільно під час роботи **`5.7 GiB` (близько 6.1 ГБ)** | Команди `sudo dmidecode -t memory` (2 модулі по `8 GB DDR4 3200 MT/s`) та `free -h` (`available: 5.7Gi`) |
| **Відеокарта (GPU) і VRAM** | Інтегрована **AMD Radeon Vega 7 (Cezanne `[1002:1638]`)**, драйвер `amdgpu`; власної GDDR6 немає, використовує близько 2 ГБ спільної RAM (UMA) | Команда `lspci -nnk | grep -iEA3 "vga|3d|display"` та різниця між 16 ГБ фізичної RAM і `14Gi` у `free -h` |
| **Пропускна здатність пам'яті** | **51.2 ГБ/с** (теоретичний пік двоканальної DDR4-3200); **від 38 до 43 ГБ/с** (реальна стала швидкість читання CPU) | Розрахунок за специфікацією [AMD Ryzen 5 5600H](https://www.cpu-monkey.com/en/cpu-amd_ryzen_5_5600h): <br>$$3200\text{ МТ/с} \times 8\text{ Б} \times 2\text{ канали} = 51.2\text{ ГБ/с}$$

 |

---

### 1.2. Розрахунок бюджету пам'яті та стелі швидкості генерації

#### 1. Оцінка пам'яті за формулами з лекції 5 ([Modular LLM Handbook](https://handbook.modular.com/getting-started/calculating-gpu-memory-for-llms/))

Загальна потреба в пам'яті для локального запуску моделі складається з ваг моделі, KV-кешу для контексту та буферів рантайму (від 10% до 30% запасу):


$$M_{\text{заг}} = M_{\text{ваг}} + M_{\text{KV}} + M_{\text{рантайм}} \approx P \times \frac{Q}{8} \times (1 + \text{Overhead})$$

На нашому ноутбуці фізично встановлено 16 ГБ RAM, з яких близько 2 ГБ зарезервовано під iGPU Vega, а з урахуванням запущеної ОС та браузера (`free -h`) доступно **`available = 5.7 GiB` (близько 6.1 ГБ)**. Щоб система не виходила у повільний дисковий `swap`, модель разом із KV-кешем (для довжини контексту 2048–4096 токенів) та буферами рантайму має вкладатися у межі до 4.5–5.0 ГБ (оптимально — до 1.5–3.5 ГБ).

#### 2. Оцінка теоретичної стелі швидкості генерації ([Martin Alderson — Local LLM Speed Calculator](https://martinalderson.com/posts/local-llm-speed-calculator/))

Оскільки генерація токенів (decode) упирається у швидкість читання ваг із оперативної пам'яті DDR4-3200, теоретична стеля швидкості становить:


$$\text{Max tokens/s (стеля)} = \frac{\text{Memory Bandwidth (51.2 ГБ/с)}}{\text{Model Weights (ГБ)}}$$


Реальна швидкість на процесорі в рантаймах `Ollama` / `llama.cpp` становить **60–80% від теоретичної стелі** (через реальну пропускну здатність DDR4 близько 38–43 ГБ/с та витрати часу CPU на обчислення уваги й деквантизацію):

| Модель та квантизація | Розмір ваг, ГБ | З урахуванням KV-кешу (ctx 2048–4096) і рантайму (+25%), ГБ | Чи вміщується у вільні 5.7 GiB (близько 6.1 ГБ)? | Теоретична стеля (при 51.2 ГБ/с), tokens/s | Очікувана реальна швидкість (60–80% стелі), tokens/s |
| --- | --- | --- | --- | --- | --- |
| `Qwen3-8B` (`fp16`, 16 біт) | 16.40 | близько 20.50 | Ні (перевищує всю фізичну RAM) | 3.1 | — |
| `Qwen3-8B` (`Q4_K_M`, близько 4.8 біта) | 5.00 | близько 6.25 | Ні / На межі (піде у swap при відкритому браузері) | 10.2 | від 6 до 8 |
| `Qwen3-4B` (`fp16`, 16 біт) | 8.04 | близько 10.05 | Ні (перевищує доступні 5.7 GiB) | 6.4 | — |
| `Qwen3-4B` (`Q4_K_M`, близько 4.8 біта) | 2.50 | близько 3.15 | Так, але займає понад 50% вільної RAM | 20.5 | від 12 до 16 |
| **`qwen3:1.7b` (`Q4_K_M`, базовий вибір)** | **1.28** | **близько 1.65** | **Так, із великим запасом (навіть для 4 паралельних слотів)** | **40.0** | **від 24 до 32** |
| **`qwen3:0.6b` (`Q4_K_M`, для оптимізації)** | **0.52** | **близько 0.75** | **Так, мінімальне навантаження на RAM** | **98.5** | **від 55 до 75** |

---

### 1.3. Обрана модель, квантизація й рантайм і чому

1. **Обраний рантайм:** **[Ollama](https://docs.ollama.com/)** — під капотом використовує оптимізований під C/C++ та векторні інструкції AVX2 рушій [llama.cpp](https://github.com/ggml-org/llama.cpp), що ідеально підходить для інференсу на процесорі AMD Ryzen 5 5600H (без дискретної відеокарти NVIDIA), автоматично керує життєвим циклом моделей і одразу надає OpenAI-сумісний HTTP API (`http://localhost:11434/v1`).
2. **Початкова модель та формат (`before`):** **[`qwen3:1.7b`](https://ollama.com/library/qwen3)** ([Qwen/Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B)) у форматі **GGUF із блоковою квантизацією `Q4_K_M**` (ваги займають близько 1.28 ГБ, а разом із KV-кешем і буферами рантайму — близько 1.65 ГБ, що легко вміщується у вільні 5.7 GiB RAM).
3. **Instruct чи Base:** Обрано **Instruct**-версію, оскільки застосунок має чітко виконувати інструкції користувача (генерувати картки для повторення, виділяти тези конспекту, пояснювати помилки коду).
4. **Чи потрібен режим міркування (`Thinking`):** **Ні (`reasoning_effort = "none"` / `think = False`)**. На процесорі генерація 200–350 прихованих токенів міркування перед кожною відповіддю збільшує затримку до появи першого видимого слова (**TTFT**) на 8–15 секунд. Вимкнення міркування дозволяє отримувати перший токен уже за 0.4–1.2 с.

---

### 1.4. Вимірювання до оптимізації (`before`)

Команда запуску навантажувального тесту для початкової конфігурації (`qwen3:1.7b`, стандартний запуск `ollama serve` з `OLLAMA_NUM_PARALLEL=1`, вимкнене міркування `--reasoning-effort none`, 10 власних промптів із `app/prompts_10.txt`):

```bash
python3 app/bench.py --base-url http://localhost:11434/v1 --model qwen3:1.7b --reasoning-effort none --prompts app/prompts_10.txt --label before --output bench_results.csv

```

**Порівняння виміряної швидкості з теоретичним прогнозом із п. 1.2 та пояснення різниці:**

* Теоретична стеля для `qwen3:1.7b` (1.28 ГБ ваг при піковій пропускній здатності DDR4-3200 51.2 ГБ/с) становить **`40.0 tokens/s`**, а реалістичний прогноз (60–80% від стелі) — **`24–32 tokens/s`**.
* Фактична швидкість генерації для 1 клієнта на CPU завжди нижча за пікову стелю `40.0 tokens/s`, тому що:
1. Реальна досяжна пропускна здатність контролера пам'яті DDR4 на мобільному процесорі Zen 3 становить від 38 до 42 ГБ/с (близько 75–80% від теоретичних 51.2 ГБ/с), причому частину смуги пропускання споживають фонові процеси ОС та вбудована графіка Radeon Vega;
2. У `qwen3:1.7b` ембединги та вихідний шар (`lm_head` на 151 936 токенів словника) зберігаються у вищій точності, а на кожному кроці процесор додатково витрачає такти на деквантизацію блоків `Q4_K_M`, обчислення квадратичної уваги (`Attention`) та читання KV-кешу.


* При збільшенні кількості паралельних клієнтів (`concurrency = 2, 4, 8`) зі стандартним `OLLAMA_NUM_PARALLEL=1` запити стають у внутрішню чергу й обробляються послідовно, тому **TTFT p50 та p95** зростають майже лінійно (кожен наступний клієнт чекає завершення генерації попередніх).

---

### 1.5. Оптимізація і вимірювання після (`after`)

Для покращення швидкодії на нашому залізі виконано дві оптимізації:

1. **Оптимізація рантайму (`after_parallel4`):** перезапуск сервера Ollama з підтримкою 4 паралельних слотів неперервного батчингу та компактним контекстом:
```bash
OLLAMA_NUM_PARALLEL=4 OLLAMA_CONTEXT_LENGTH=2048 ollama serve
python3 app/bench.py --base-url http://localhost:11434/v1 --model qwen3:1.7b --reasoning-effort none --prompts app/prompts_10.txt --label after_parallel4 --output bench_results.csv

```


Завдяки батчингу ваги моделі читаються з RAM один раз одночасно для кількох активних запитів, що суттєво знижує **TTFT p50/p95** при 2–4 одночасних клієнтах і підвищує сумарний **Throughput** сервера.
2. **Оптимізація розміру моделі (`after_0.6b`):** перехід на молодшу модель того ж сімейства **`qwen3:0.6b`** (`Q4_K_M`, обсяг ваг лише близько 0.52 ГБ):
```bash
python3 app/bench.py --base-url http://localhost:11434/v1 --model qwen3:0.6b --reasoning-effort none --prompts app/prompts_10.txt --label after_0.6b --output bench_results.csv

```


Завдяки зменшенню обсягу ваг у 2.46 раза швидкість генерації `tokens/s` на одного користувача та загальний `throughput` зростають майже вдвічі, а `TTFT` скорочується до часток секунди. Натомість при перевірці на 10 власних промптах (`app/prompts_10.txt`) модель `qwen3:1.7b` дає помітно грамотнішу українську мову та точніші картки, ніж `qwen3:0.6b`, тому в застосунку залишено можливість миттєвого перемикання між `qwen3:1.7b` (максимальна якість українською) та `qwen3:0.6b` (максимальна швидкість).

In [ ]:
# Автоматичне відображення результатів із bench_results.csv (після запуску bench.py)
import os
if os.path.exists("bench_results.csv"):
    df_bench = pd.read_csv("bench_results.csv")
    display(df_bench)
else:
    print("Файл bench_results.csv буде згенеровано після запуску bench.py з мітками before та after.")

---

### 1.6. Власний застосунок — що робить, як працює та параметри генерації

Розроблено мінімалістичний застосунок **«Навчальний AI-Помічник (Конспекти, Картки та Код)»** у теці `/lab4/app/`:
* **Вебінтерфейс (`app/index.html`)** — чиста односторінкова HTML/CSS/JS-програма (без зовнішніх фреймворків чи окремого бекенду), яка напряму звертається до локального сервера через OpenAI-сумісний ендпойнт `POST /v1/chat/completions` (`stream: true`).
* **Термінальний інтерфейс (`app/cli.py`)** — консольний Python-клієнт на базі бібліотеки `openai` для роботи з файлами конспектів (`--file`) або інтерактивного діалогу (`--interactive`).

**Що робить застосунок (4 спеціалізовані режими системного промпту):**
1. **Картки для повторення (`flashcards`)** — перетворює конспект лекції або тему на структуровані картки `Питання — Відповідь`.
2. **Стислий конспект (`summary`)** — виділяє з навчального тексту рівно 3–5 головних тез маркованим списком.
3. **Пояснення помилки в коді або логах (`code_review`)** — аналізує текст помилки (наприклад, CUDA/PyTorch traceback) і пропонує виправлення.
4. **Вільний запит (`chat`)** — прямий діалог із локальною моделлю.

**Обґрунтування обраних параметрів генерації:**
* **`temperature = 0.2`**: низька температура забезпечує високу фактичну точність під час конспектування та генерації коду, мінімізуючи лексичні галюцинації малої моделі.
* **`max_tokens = 400` (обов'язковий ліміт токенів відповіді)**: слугує жорстким запобіжником від нескінченного зациклення малої моделі (repetition loop) і гарантує стислість карток.
* **`reasoning_effort = "none"` (`think = False`)**: вимикає приховані токени міркування в `Qwen3`, зменшуючи затримку до першого видимого слова (`TTFT`) на процесорі у 5–10 разів.
* **`num_ctx = 2048` (довжина контексту)**: достатня для конспекту однієї теми (~1500 слів) і водночас береже оперативну пам'ять під KV-кеш.
* **Зворотний зв'язок та обробка помилок**: відповідь виводиться потоково (стрімінгом) у реальному часі з показом `TTFT`, повної затримки, кількості токенів і `tokens/s`; реалізовано зрозумілі повідомлення для випадків вимкненого сервера (`ConnectionError` / CORS), перевищення часу очікування (`AbortSignal.timeout`) та порожньої відповіді моделі.

---

### 1.7. Тестові сценарії й результати та розбір помилки моделі

У застосунок (`app/index.html` та скрипт `app/test_scenarios.py`) вбудовано **8 тестових сценаріїв**, що охоплюють типові, складні та крайові випадки:

| № | Назва та тип сценарію | Короткий опис вхідного запиту | Що вважається правильним результатом | Результат виконання |
|---|---|---|---|---|
| **1** | **Типовий:** Генерація карток із конспекту | Конспект про формати `fp16`, `bf16`, `int8`, `int4`, `int2` у режимі `flashcards` | 3–5 чітких карток у форматі «Питання — Відповідь» без видуманих фактів | **Успішно:** модель генерує структуровані картки з точним описом розрядності |
| **2** | **Типовий:** Стислий конспект | Абзац про метрики `TTFT`, `TPOT (tokens/s)`, `Throughput` та перцентилі `p50`/`p95` у режимі `summary` | 4 лаконічні марковані пункти українською мовою | **Успішно:** усі 4 метрики виділено короткими тезами |
| **3** | **Типовий:** Аналіз логу помилки GPU | Лог `ValueError: Bfloat16 is only supported on GPUs with compute capability of at least 8.0... T4 has 7.5` | Пояснення, що T4 (7.5) не підтримує `bf16` апаратно, і порада змінити тип на `torch.float16` (`fp16`) | **Успішно:** модель точно вказує на заміну `bfloat16 -> float16` |
| **4** | **Типовий:** Генерація функції Python | Написати функцію `percentile(values, q)` без `numpy` | Робочий код Python із сортуванням масиву та лінійною інтерполяцією | **Успішно:** згенеровано коректну функцію та приклад виклику |
| **5** | **Складний:** Багатокроковий розрахунок заліза | Розрахувати пам'ять із запасом 30% (`1.28 ГБ * 1.3`) та стелю швидкості (`51.2 / 1.28`) для Ryzen 5 5600H | Точні числа: `1.664 ГБ` (< 5.7 ГіБ, вміщується) та стеля `40 tokens/s` | **Успішно:** обидві дії множення та ділення виконано правильно |
| **6** | **Крайовий:** Запит із хибною передумовою | Питання-пастка: «Чому при `temperature = 0` модель починає максимально фантазувати?» | Модель не погоджується з помилковим твердженням, а пояснює, що `temperature = 0` навпаки вимикає випадковість | **Успішно:** модель коректно спростовує хибне припущення користувача |
| **7** | **Крайовий:** Перевірка запобіжника `max_tokens` | Прохання виписати всі числа від 1 до 500 словами при ліміті `max_tokens = 80` | Генерація безпечно обривається на 80 токенах зі статусом `[Зупинено лімітом max_tokens=80]` без зависання | **Успішно:** ліміт токенів коректно зупиняє нескінченний перелік |
| **8** | **Крайовий:** Посимвольний підрахунок літери «р» | Підрахунок літери «р» у слові «трансформер» | Точна відповідь: **3** літери «р» (на позиціях 2, 8, 12) | **Виправлено після модифікації промпту** (див. розбір нижче) |

#### Розбір ситуації, де модель помилилася, та як її було виправлено
1. **Ситуація 1 (Порожня відповідь або обрив через приховане міркування):**
   * *У чому полягала помилка:* Під час перших запусків `qwen3:1.7b` із лімітом `max_tokens = 200` без параметра `reasoning_effort` модель за замовчуванням вмикала режим міркування (`thinking`), витрачала всі 200 токенів на внутрішні роздуми англійською мовою і повертала порожнє поле `content`, через що клієнт видавав помилку порожньої відповіді.
   * *Що зробили для виправлення:* Додали явну передачу параметра `reasoning_effort: "none"` у запит `/v1/chat/completions` (та вибір цього режиму за замовчуванням у UI застосунку), а також інформативне повідомлення про помилку, якщо `content` порожній. Після цього модель одразу генерує видиму відповідь із низьким `TTFT`.
2. **Ситуація 2 (Помилка підрахунку літер у цілому слові — Сценарій 8):**
   * *У чому полягала помилка:* На прямому запиті *«Скільки разів літера «р» трапляється в слові «трансформер»?»* модель `qwen3:1.7b` відповідала **«2 рази»**, оскільки бачила слово як кілька склеєних BPE-токенів.
   * *Що зробили для виправлення:* Змінили формулювання промпту — подали слово з розділенням літер дефісами (`«т - р - а - н - с - ф - о - р - м - е - р»`) і попросили спочатку пронумерувати кожну літеру списком, а потім дати підсумок. Оскільки кожна літера стала окремим токеном, модель безпомилково знайшла всі **3** входження літери «р».

---

### 1.8. Висновок

У ході виконання лабораторної роботи досліджено особливості локального запуску та оптимізації відкритих мовних моделей:
1. На прикладі сімейства **Qwen3** експериментально перевірено різницю між **Base**, **Instruct** та **Thinking** моделями, а також вплив квантизації (`fp16`, `int8`, `int4`, `int2`) на обсяг пам'яті, швидкість генерації, перплексію та якість перекладу (`chrF`, `BLEU`). Підтверджено, що 8-бітна та 4-бітна квантизація дозволяють скоротити споживання пам'яті у ~1.8–3.1 раза майже без втрати якості, тоді як 2-бітна квантизація (`int2`) руйнує когерентність генерації, а менші моделі (`1.7B` проти `4B`) та українська мова є значно чутливішими до агресивного стиснення ваг.
2. Виконано інженерний розрахунок бюджету заліза для ноутбука **ASUS VivoBook M3502QA** (процесор **AMD Ryzen 5 5600H**, **16 ГБ DDR4-3200 Dual-Channel** із піковою пропускною здатністю **51.2 ГБ/с** та **5.7 GiB** доступної RAM), обґрунтовано вибір рантайму **Ollama** та моделей **`qwen3:1.7b`** і **`qwen3:0.6b`** у форматі GGUF (`Q4_K_M`), а також реалізовано й протестовано власний мінімалістичний веб- та CLI-застосунок **«Навчальний AI-Помічник»**.